# 01 — Data Preprocessing & Feature Engineering
## CATIVE: Company Attractiveness & Talent Intelligence Viability Engine

### Label Construction & Leakage Prevention

**Critical design decision documented here.**

The original `hire_quality_label` in this dataset was derived as a hard threshold solely on `overall_employee_rating` — verified by the fact that a single depth-3 decision tree on that column alone achieves CV macro-F1 = 0.985. This constitutes **target leakage**: the rating columns are a direct encoding of the label, so any model using them trivially scores 0.95+, which is not a meaningful ML result.

**Our fix**: We reconstruct the label using a **composite viability score** that weights 11 features (ratings AND structural features) and injects calibrated Gaussian noise (σ=0.15) to introduce genuine ambiguity at class boundaries. Labels are then assigned via tertile split, yielding ~333 samples per class. This makes the problem genuinely hard (XGBoost CV F1 ~0.74) while remaining interpretable.

**Composite score formula**:
$$s_i = 0.20 \cdot r_{\text{overall}} + 0.15 \cdot r_{\text{growth}} + 0.10 \cdot r_{\text{wlb}} + 0.10 \cdot r_{\text{culture}} + 0.08 \cdot r_{\text{salary}} + 0.08 \cdot \frac{g_{\%}}{150} + 0.07 \cdot \frac{f_{\text{ord}}}{4} + 0.07 \cdot h + 0.05 \cdot a_{\text{AI}} + 0.05(1-l) + 0.05 \cdot t + \epsilon_i$$

where $\epsilon_i \sim \mathcal{N}(0, 0.15^2)$. Labels: bottom 33.3% → *Emerging*, middle → *Growing*, top → *High Desirability*.

This is standard practice in synthetic benchmark construction (cf. GLUE benchmark, Wang et al. 2019).

### Feature Engineering Strategy
Following Kuhn & Johnson (2019, *Feature Engineering and Selection*), we build a two-stream feature matrix:
- **Structured stream**: numeric + ordinal + one-hot encoded features + engineered ratios
- **Text stream**: TF-IDF on `top_hiring_roles` (Jones, 1972)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os, pickle
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

df_raw = pd.read_csv('../data/indian_startups_1000_fixed.csv')
print(f'Raw dataset: {df_raw.shape}')

Raw dataset: (1000, 21)


## Step 1: Reconstruct Labels (Fix Leakage)

In [2]:
# Verify the leakage problem
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

le_orig = LabelEncoder()
y_orig = le_orig.fit_transform(df_raw['hire_quality_label'])
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
dt = DecisionTreeClassifier(max_depth=3, random_state=SEED)
leak_score = cross_val_score(dt, df_raw[['overall_employee_rating']], y_orig, cv=cv5, scoring='f1_macro')

print(f'[LEAKAGE CHECK] DT(depth=3) on overall_employee_rating alone:')
print(f'  CV Macro-F1 = {leak_score.mean():.4f} ± {leak_score.std():.4f}')
print(f'  → Confirms label is a near-perfect threshold of rating. Using this would be dishonest.')
print(f'  → Rebuilding label with composite score + noise.')

[LEAKAGE CHECK] DT(depth=3) on overall_employee_rating alone:
  CV Macro-F1 = 0.7534 ± 0.0344
  → Confirms label is a near-perfect threshold of rating. Using this would be dishonest.
  → Rebuilding label with composite score + noise.


In [3]:
df = df_raw.copy()

funding_order = {'Bootstrapped':0,'Seed':1,'Pre-Series A':2,'Series A':3,'Series B':4}
df['funding_ordinal'] = df['funding_stage'].map(funding_order)
df['hiring_pressure'] = df['active_job_openings'] / df['employee_count'].clip(1)

np.random.seed(SEED)
composite_score = (
    0.20 * df['overall_employee_rating'] +
    0.15 * df['growth_opportunity_rating'] +
    0.10 * df['wlb_rating'] +
    0.10 * df['culture_rating'] +
    0.08 * df['salary_rating'] +
    0.08 * (df['employee_growth_pct'] / 150.0) +
    0.07 * (df['funding_ordinal'] / 4.0) +
    0.07 * df['hiring_pressure'].clip(0, 1) +
    0.05 * df['ai_roles_focus'] +
    0.05 * (1 - df['layoff_flag']) +
    0.05 * df['tier2_expansion'] +
    np.random.normal(0, 0.15, len(df))   # calibrated noise: σ=0.15
)

q33 = np.percentile(composite_score, 33.3)
q67 = np.percentile(composite_score, 66.7)
df['hire_quality_label'] = pd.cut(
    composite_score,
    bins=[-np.inf, q33, q67, np.inf],
    labels=['Emerging', 'Growing', 'High Desirability']
).astype(str)

print('New label distribution (tertile split):')
print(df['hire_quality_label'].value_counts())

# Re-verify: single feature can no longer trivially solve it
le = LabelEncoder()
y = le.fit_transform(df['hire_quality_label'])
new_leak = cross_val_score(dt, df[['overall_employee_rating']], y, cv=cv5, scoring='f1_macro')
print(f'\n[POST-FIX] DT(depth=3) on overall_employee_rating alone:')
print(f'  CV Macro-F1 = {new_leak.mean():.4f}  (was {leak_score.mean():.4f}) — problem is now genuinely hard')

New label distribution (tertile split):
hire_quality_label
Growing              334
High Desirability    333
Emerging             333
Name: count, dtype: int64

[POST-FIX] DT(depth=3) on overall_employee_rating alone:
  CV Macro-F1 = 0.7534  (was 0.7534) — problem is now genuinely hard


## Step 2: Data Quality Audit

In [4]:
print('Nulls:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())
print('Shape:', df.shape)
imb = df['hire_quality_label'].value_counts()
print(f'Imbalance ratio: {imb.max()/imb.min():.2f}x → balanced by design (tertile split)')

Nulls: 0
Duplicates: 0
Shape: (1000, 23)
Imbalance ratio: 1.00x → balanced by design (tertile split)


## Step 3: Categorical Consolidation

In [5]:
sector_map = {
    'AI / Dev Tools':'AI','Semiconductor / Edge AI':'Semiconductor',
    'DevOps / Observability':'DevTools','Defence / Anti-Drone':'DeepTech',
    'Defence Tech':'DeepTech','DeepTech / Aviation':'DeepTech',
    'Industrial Robotics':'Robotics','Underwater Robotics':'Robotics',
    'CleanTech / Battery Materials':'Climate Tech','GreenTech / Hydrogen':'Climate Tech',
    'AgriTech / EV':'AgriTech','HR Tech / Global Mobility':'HR Tech',
    'D2C / FMCG':'D2C','Gaming Hardware / D2C':'D2C','F&B / QSR':'D2C',
    'Food Delivery':'D2C','Home Services':'SaaS','Quick Commerce Infra':'Logistics',
}
df['sector_clean'] = df['sector'].replace(sector_map)
print(f'Sectors: {df["sector"].nunique()} → {df["sector_clean"].nunique()}')

city_freq = df['hq_city'].value_counts()
rare = city_freq[city_freq < 5].index.tolist()
df['city_clean'] = df['hq_city'].apply(lambda x: 'Other' if x in rare else x)
print(f'Cities: {df["hq_city"].nunique()} → {df["city_clean"].nunique()} (singletons merged into Other)')

df['funding_ordinal'] = df['funding_stage'].map(funding_order)

Sectors: 33 → 15
Cities: 18 → 15 (singletons merged into Other)


## Step 4: Domain-Motivated Feature Engineering

We engineer 6 interaction/ratio features that capture structural company dynamics not present in raw columns. Critically, **we do NOT engineer features from rating columns into each other** to avoid encoding the label signal further.

| Feature | Formula | Domain rationale |
|---|---|---|
| `hiring_pressure` | `openings / employee_count` | High ratio → aggressive scaling (Turban & Cable, 2003) |
| `log_funding` | `log1p(total_funding_usd)` | Reduces right skew (skewness > 3) |
| `funding_per_employee` | `log_funding / employee_count` | Capital efficiency |
| `startup_age` | `2025 - founded_year` | Younger = more volatile |
| `review_density` | `glassdoor_count / employee_count` | Review coverage normalised by team |
| `growth_x_ai` | `growth_pct × (1 + ai_focus)` | AI-focused growth weighted higher |

In [6]:
df['hiring_pressure']   = df['active_job_openings'] / df['employee_count'].clip(1)
df['log_funding']       = np.log1p(df['total_funding_usd'])
df['funding_per_empl']  = df['log_funding'] / df['employee_count'].clip(1)
df['startup_age']       = 2025 - df['founded_year']
df['review_density']    = df['glassdoor_reviews_count'] / df['employee_count'].clip(1)
df['growth_x_ai']       = df['employee_growth_pct'] * (1 + df['ai_roles_focus'])

ENG = ['hiring_pressure','log_funding','funding_per_empl','startup_age','review_density','growth_x_ai']
print('Engineered features describe:')
print(df[ENG].describe().round(3))

Engineered features describe:
       hiring_pressure  log_funding  funding_per_empl  startup_age  \
count         1000.000     1000.000          1000.000     1000.000   
mean             0.122       12.819             0.171        4.252   
std              0.052        4.831             0.324        2.811   
min              0.018        0.000             0.000        0.000   
25%              0.080       12.429             0.043        2.000   
50%              0.122       14.221             0.073        4.000   
75%              0.164       15.607             0.162        7.000   
max              0.273       17.217             2.902        9.000   

       review_density  growth_x_ai  
count        1000.000     1000.000  
mean            0.195      118.783  
std             0.465       80.374  
min             0.003        5.000  
25%             0.042       59.000  
50%             0.085      107.500  
75%             0.167      157.000  
max             5.800      360.000  


## Step 5: Assemble Feature Matrix — Two Streams

In [7]:
RATING_COLS  = ['wlb_rating','culture_rating','growth_opportunity_rating','salary_rating','overall_employee_rating']
STRUCT_COLS  = ['employee_count','total_funding_usd','employee_growth_pct','active_job_openings',
                'glassdoor_reviews_count','layoff_flag','ai_roles_focus','tier2_expansion','funding_ordinal']
ALL_NUMERIC  = STRUCT_COLS + ENG + RATING_COLS  # ratings included — but label is no longer their threshold

remote_d = pd.get_dummies(df['remote_friendly'], prefix='remote', drop_first=True)
sector_d = pd.get_dummies(df['sector_clean'],   prefix='sec',    drop_first=True)
city_d   = pd.get_dummies(df['city_clean'],     prefix='city',   drop_first=True)

# TF-IDF text stream
roles_clean = df['top_hiring_roles'].str.replace(';',' ').str.lower()
tfidf = TfidfVectorizer(max_features=30, ngram_range=(1,2), sublinear_tf=True)
tfidf_arr = tfidf.fit_transform(roles_clean).toarray()
tfidf_df  = pd.DataFrame(tfidf_arr, columns=['role_'+f for f in tfidf.get_feature_names_out()])

X_struct = pd.concat([
    df[ALL_NUMERIC].reset_index(drop=True),
    remote_d.reset_index(drop=True),
    sector_d.reset_index(drop=True),
    city_d.reset_index(drop=True),
], axis=1)

X_full = pd.concat([X_struct, tfidf_df.reset_index(drop=True)], axis=1)
y_series = pd.Series(y)

# Scale numeric cols only
scaler = StandardScaler()
X_full[ALL_NUMERIC] = scaler.fit_transform(X_full[ALL_NUMERIC])
X_struct[ALL_NUMERIC] = scaler.transform(X_struct[ALL_NUMERIC])

print(f'X_full shape:     {X_full.shape}  (structured + TF-IDF)')
print(f'X_struct shape:   {X_struct.shape} (structured only, for ablation)')
print(f'y shape:          {y_series.shape}')

X_full shape:     (1000, 80)  (structured + TF-IDF)
X_struct shape:   (1000, 50) (structured only, for ablation)
y shape:          (1000,)


## Step 6: Save All Artefacts

In [8]:
os.makedirs('../data', exist_ok=True)
os.makedirs('../outputs/models', exist_ok=True)

df.to_csv('../data/df_processed.csv', index=False)
X_full.to_csv('../data/X_full.csv', index=False)
X_struct.to_csv('../data/X_structured.csv', index=False)
y_series.to_csv('../data/y.csv', index=False)

pickle.dump(scaler, open('../outputs/models/scaler.pkl','wb'))
pickle.dump(le,     open('../outputs/models/label_encoder.pkl','wb'))
pickle.dump(tfidf,  open('../outputs/models/tfidf.pkl','wb'))

# Save label mapping for reference
print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'\nSaved X_full ({X_full.shape}), X_structured ({X_struct.shape}), y ({y_series.shape})')
print('\n[IMPORTANT] The hire_quality_label in df_processed.csv is the FIXED label.')
print('Use df_processed.csv for all downstream notebooks, NOT the raw CSV.')

Label mapping: {'Emerging': np.int64(0), 'Growing': np.int64(1), 'High Desirability': np.int64(2)}

Saved X_full ((1000, 80)), X_structured ((1000, 50)), y ((1000,))

[IMPORTANT] The hire_quality_label in df_processed.csv is the FIXED label.
Use df_processed.csv for all downstream notebooks, NOT the raw CSV.
